In [0]:
%pip install -q umap-learn hdbscan scikit-learn plotly

import re
import os
import numpy as np
import pandas as pd
import plotly.express as px

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.feature_extraction import text as sk_text
from sklearn.feature_extraction.text import TfidfVectorizer

import umap
import hdbscan


SOURCE_TABLE  = "noc_documents_azure"
OUT_TABLE     = "noc_documents_clustering"

PCA_DIMS          = 50      
UMAP_DIMS         = 15    
UMAP_NEIGHBORS    = 30
UMAP_MIN_DIST     = 0.0      

UMAP_VIZ_DIMS     = 3
UMAP_VIZ_MIN_DIST = 0.1    

MIN_CLUSTER_SIZE  = 50      
MIN_SAMPLES       = 5

TOP_WORDS         = 12
MAX_FEATURES      = 5000
MIN_DF            = 10
NGRAM_RANGE       = (1, 2)

REASSIGN_NOISE    = True     

VIZ_SAMPLE_FRAC   = 0.5


stop_en = set(sk_text.ENGLISH_STOP_WORDS)

stop_fr = {
    "alors","au","aucuns","aussi","autre","avant","avec","avoir","bon","car",
    "ce","cela","ces","ceux","chaque","ci","comme","comment","dans","des","du",
    "dedans","dehors","depuis","devrait","doit","donc","dos","début","elle",
    "elles","en","encore","essai","est","et","eu","fait","faites","fois","font",
    "force","haut","hors","ici","il","ils","je","juste","la","le","les","leur",
    "là","ma","maintenant","mais","mes","mine","moins","mon","mot","même","ni",
    "nommés","notre","nous","nouveaux","ou","où","par","parce","parole","pas",
    "personnes","peut","peu","pièce","plupart","pour","pourquoi","quand","que",
    "quel","quelle","quelles","quels","qui","sa","sans","ses","seulement","si",
    "sien","son","sont","sous","soyez","sujet","sur","ta","tandis","tellement",
    "tels","tes","ton","tous","tout","trop","très","tu","valeur","voie","voient",
    "vont","votre","vous","vu","ça","étaient","état","étions","été","être",
    "un","une","aux","toute","toutes","tous","tout","autre","autres","même",
    "aussi","alors","après","avant","comme","entre","sans","sous","lors","selon",
    "faire","dit","sont","ont","était","ainsi","cas","doit","peut","étaient",
    "également","émis","éléments","jusqu","cours","fois","trois","deux","quatre",
    "décrites","portant","conséquent","jours","veuillez","effets","motifs",
    "aujourd","hui",
}

domain_stopwords = {
    "drug", "canada", "health", "product", "notice", "submission", "submissions",
    "authorization", "manufacturer", "minister", "office", "bureau", "label",
    "labelling", "version", "final", "new", "used", "use", "including", "included",
    "proposed", "requested", "request", "direct", "intended", "accordance",
    "paragraph", "reminded", "connection", "basis", "document", "documents",
    "specimens", "specimen", "submit", "labels", "intellectual", "property",
    "therapeutic", "direction", "insert", "inserts", "brochures", "brochure",
    "cards", "card", "statement", "setting", "file", "cover", "page", "date",
    "mcg", "summary", "baseline", "endpoint", "table", "rate", "mean", "group",
    "groups", "overall", "analysis", "guidance", "reported", "considered",
    "provided", "issues", "approximately", "doctor", "refer", "section",
    "annex", "annexe", "appendix", "figure", "fig", "presentation", "presentations",
    # Vaccine/biologic 
    "vaccine", "vaccin", "biologic", "radiopharmaceutical", "radiopharmaceutiques",
    "drugs", "générale", "validation", "traitements", "types", "requis",
    "responsibility", "therapies", "genetic", "veterinary", "directorate",
    # Regulatory / label submission
    "toute", "ministre", "minister", "dépliant", "échantillons", "intellectuelle",
    "jointe", "emballage", "fiche", "forme", "visés", "paragraphe", "utilisées",
    "déclaration", "utilisée", "modification", "utiliser", "nécessite",
    "changement", "destinée", "abrégée", "abbreviated", "exigences", "drogues",
    "finales", "abrégées", "soumetttre", "obligations", "subsection", "require",
    "specified", "respect", "matters", "change", "products", "supplements",
    "package", "prévu", "commencer", "indiquant", "laquelle", "destinées",
    "office", "connection", "nouvelle",
    # Clinical trial boilerplate 
    "événements", "indésirables", "études", "cliniques", "étant", "vitro",
    "étapes", "virus", "outre", "rapports", "copies", "règlements", "aliments",
    "article", "présente", "soit", "thérapeutiques", "suppléments",
    # French boilerplate
    "drogue", "produit", "produits", "santé", "présentation", "présentations",
    "fabricant", "étiquette", "étiquettes", "compris", "alinéa", "conformément",
    "rappelons", "définitive", "parvenir", "prions", "soumettre", "dossier",
    "médicament", "médicaments", "propriété", "couverture", "biologiques",
    "sommaire", "examen", "médecin",
    # Add to DOMAIN_JUNK_STOPWORDS
    "sabourin", "barbara", "supriya", "sharma", "peterson", "robert",
    "sophie", "laurent", "alfred", "celia", "lourenco", "renaud", "louis",
    "omer", "boudreau", "phd", "mph", "frcpc", "bpa", "orl", "tab",
    "sipd", "trd", "crm", "attn", "signé", "signed", "verso",
    "fabriquant", "expliquées", "notification", "formule",
        
}

ALL_STOPWORDS = stop_en | stop_fr | domain_stopwords

def custom_analyzer(text):
    if not isinstance(text, str):
        return []
    s = text.lower()
    tokens = re.findall(r"[a-zàâäçéèêëîïôöùûüÿñæœ]{3,}", s)
    return [t for t in tokens if t not in ALL_STOPWORDS and not t.isdigit()]


spark = SparkSession.builder.getOrCreate()

df = (
    spark.table(SOURCE_TABLE)
         .select("path", "final_text", "embedding")
         .filter(F.col("embedding").isNotNull())
)
print(f"Docs loaded: {df.count():,}")

pdf = df.toPandas()
X = np.vstack(pdf["embedding"].values).astype(np.float32)
print(f"Embedding matrix: {X.shape}")

zero_mask = (X == 0).all(axis=1)
print(f"Zero embeddings removed: {zero_mask.sum()}")
pdf = pdf[~zero_mask].reset_index(drop=True)
X = X[~zero_mask]


X = normalize(X)

print(f"\nRunning PCA ({PCA_DIMS} dims)...")
pca = PCA(n_components=PCA_DIMS, random_state=42)
X_pca = pca.fit_transform(X)
print(f"PCA shape: {X_pca.shape}")

print(f"Running UMAP ({UMAP_DIMS} dims for clustering)...")
reducer = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    n_components=UMAP_DIMS,
    min_dist=UMAP_MIN_DIST,
    metric="cosine",
    random_state=42,
    low_memory=True
)
X_umap = reducer.fit_transform(X_pca)
print(f"UMAP shape: {X_umap.shape}")


print("\nRunning HDBSCAN...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_method="eom",
    prediction_data=True
)
labels = clusterer.fit_predict(X_umap)
pdf["cluster"] = labels
pdf["was_noise"] = (labels == -1)

n_noise    = int((labels == -1).sum())
n_total    = len(labels)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print(f"Clusters found: {n_clusters}")
print(f"Noise docs:     {n_noise} ({n_noise/n_total:.1%})")


if REASSIGN_NOISE and n_noise > 0 and n_clusters > 0:
    print("\nReassigning noise docs to nearest cluster centroid...")

    labels_arr = pdf["cluster"].values

    centroids = {}
    for cid in sorted(set(labels_arr)):
        if cid == -1:
            continue
        idx = np.where(labels_arr == cid)[0]
        cvec = X[idx].mean(axis=0)
        cvec /= (np.linalg.norm(cvec) + 1e-12)
        centroids[int(cid)] = cvec

    centroid_ids  = np.array(list(centroids.keys()), dtype=np.int32)
    centroid_mat  = np.vstack([centroids[c] for c in centroid_ids]).astype(np.float32)

    noise_idx     = np.where(labels_arr == -1)[0]
    sims          = X[noise_idx] @ centroid_mat.T
    best          = np.argmax(sims, axis=1)
    pdf.loc[noise_idx, "cluster"] = centroid_ids[best]

    print(f"Noise after reassignment: {(pdf['cluster'] == -1).sum()}")


print("\nBuilding TF-IDF topic labels...")

vectorizer = TfidfVectorizer(
    analyzer=custom_analyzer,
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE
)
tfidf = vectorizer.fit_transform(pdf["final_text"].astype(str).values)
terms = np.array(vectorizer.get_feature_names_out())

cluster_label_map = {}
for cid in sorted(set(pdf["cluster"].values)):
    if cid == -1:
        continue
    idx        = np.where(pdf["cluster"].values == cid)[0]
    mean_tfidf = np.asarray(tfidf[idx].mean(axis=0)).ravel()
    top_idx    = np.argsort(mean_tfidf)[-TOP_WORDS:][::-1]
    cluster_label_map[int(cid)] = ", ".join(terms[top_idx])

pdf["cluster_keywords"] = pdf["cluster"].map(
    lambda c: cluster_label_map.get(int(c), "noise")
)

print("\nTop keywords per cluster:")
for cid, kw in sorted(cluster_label_map.items()):
    count = (pdf["cluster"] == cid).sum()
    print(f"  Cluster {cid:>3} ({count:>5} docs): {kw}")


print(f"\nRunning UMAP 3D for visualization...")
reducer_3d = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    n_components=UMAP_VIZ_DIMS,
    min_dist=UMAP_VIZ_MIN_DIST,
    metric="cosine",
    random_state=42,
    low_memory=True
)
X_3d = reducer_3d.fit_transform(X_pca)  

pdf["x"] = X_3d[:, 0]
pdf["y"] = X_3d[:, 1]
pdf["z"] = X_3d[:, 2]


plot_df = (
    pdf.groupby("cluster", group_keys=False)
       .apply(lambda g: g.sample(frac=VIZ_SAMPLE_FRAC, random_state=42),
              include_groups=False)
       .reset_index(drop=True)
)
for col in ["cluster", "x", "y", "z", "cluster_keywords", "was_noise"]:
    plot_df[col] = pdf.loc[plot_df.index, col].values

plot_df["preview"]     = pdf.loc[plot_df.index, "final_text"].str[:100].str.replace("\n", " ").values
plot_df["cluster_str"] = "Cluster " + plot_df["cluster"].astype(str)
plot_df["rescued"]     = plot_df["was_noise"].map({True: "Yes (rescued noise)", False: "No"})

print(f"Sampled {len(plot_df):,} docs for visualization")

fig = px.scatter_3d(
    plot_df,
    x="x", y="y", z="z",
    color="cluster_str",
    hover_data={
        "cluster_str": True,
        "cluster_keywords": True,
        "rescued": True,
        "preview": True,
        "x": False, "y": False, "z": False,
    },
    title=f"NOC Document Clusters — {n_clusters} clusters · {len(pdf):,} docs",
    labels={"cluster_str": "Cluster"},
    opacity=0.65,
    height=850,
)
fig.update_traces(marker=dict(size=2.5))
fig.update_layout(
    scene=dict(
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        zaxis_title="UMAP-3",
        bgcolor="rgb(15,17,23)",
    ),
    paper_bgcolor="rgb(15,17,23)",
    font_color="white",
    legend_title_text="Cluster",
)

try:
    os.makedirs("/dbfs/tmp", exist_ok=True)
    html_path = "/dbfs/tmp/cluster_viz_3d.html"
    fig.write_html(html_path)
    print(f" 3D viz saved: {html_path}")
except Exception:
    html_path = "/tmp/cluster_viz_3d.html"
    fig.write_html(html_path)
    print(f" 3D viz saved (local): {html_path}")
    print("   Copy to DBFS: dbutils.fs.cp('file:/tmp/cluster_viz_3d.html', 'dbfs:/tmp/cluster_viz_3d.html')")

displayHTML(fig.to_html(full_html=False, include_plotlyjs="cdn"))


out_pdf = pdf[["path", "cluster", "cluster_keywords", "was_noise"]].copy()
out_sdf = spark.createDataFrame(out_pdf)
out_sdf.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(OUT_TABLE)
print(f"\n Results saved → {OUT_TABLE}")


spark.table(OUT_TABLE) \
     .groupBy("cluster", "cluster_keywords") \
     .agg(F.count("*").alias("doc_count")) \
     .orderBy(F.desc("doc_count")) \
     .show(200, truncate=False)



In [0]:
%pip install -q umap-learn hdbscan scikit-learn plotly

import re
import os
import numpy as np
import pandas as pd
import plotly.express as px

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.feature_extraction import text as sk_text
from sklearn.feature_extraction.text import TfidfVectorizer

import umap
import hdbscan

SOURCE_TABLE  = "noc_documents_qwen3_06b"
OUT_TABLE     = "noc_documents_clustering6"

PCA_DIMS          = 50
UMAP_DIMS         = 15
UMAP_NEIGHBORS    = 30
UMAP_MIN_DIST     = 0.0

UMAP_VIZ_DIMS     = 3
UMAP_VIZ_MIN_DIST = 0.1

MIN_CLUSTER_SIZE  = 50
MIN_SAMPLES       = 5

TOP_WORDS         = 12
MAX_FEATURES      = 5000
MIN_DF            = 10
NGRAM_RANGE       = (1, 2)

REASSIGN_NOISE    = True
VIZ_SAMPLE_FRAC   = 0.5

EMBED_LEN_MODE   = True
EMBED_LEN_FORCE  = None   

stop_en = set(sk_text.ENGLISH_STOP_WORDS)

stop_fr = {
    "alors","au","aucuns","aussi","autre","avant","avec","avoir","bon","car",
    "ce","cela","ces","ceux","chaque","ci","comme","comment","dans","des","du",
    "dedans","dehors","depuis","devrait","doit","donc","dos","début","elle",
    "elles","en","encore","essai","est","et","eu","fait","faites","fois","font",
    "force","haut","hors","ici","il","ils","je","juste","la","le","les","leur",
    "là","ma","maintenant","mais","mes","mine","moins","mon","mot","même","ni",
    "nommés","notre","nous","nouveaux","ou","où","par","parce","parole","pas",
    "personnes","peut","peu","pièce","plupart","pour","pourquoi","quand","que",
    "quel","quelle","quelles","quels","qui","sa","sans","ses","seulement","si",
    "sien","son","sont","sous","soyez","sujet","sur","ta","tandis","tellement",
    "tels","tes","ton","tous","tout","trop","très","tu","valeur","voie","voient",
    "vont","votre","vous","vu","ça","étaient","état","étions","été","être",
    "un","une","aux","toute","toutes","tous","tout","autre","autres","même",
    "aussi","alors","après","avant","comme","entre","sans","sous","lors","selon",
    "faire","dit","sont","ont","était","ainsi","cas","doit","peut","étaient",
    "également","émis","éléments","jusqu","cours","fois","trois","deux","quatre",
    "décrites","portant","conséquent","jours","veuillez","effets","motifs",
    "aujourd","hui",
}

domain_stopwords = {
    "drug", "canada", "health", "product", "notice", "submission", "submissions",
    "authorization", "manufacturer", "minister", "office", "bureau", "label",
    "labelling", "version", "final", "new", "used", "use", "including", "included",
    "proposed", "requested", "request", "direct", "intended", "accordance",
    "paragraph", "reminded", "connection", "basis", "document", "documents",
    "specimens", "specimen", "submit", "labels", "intellectual", "property",
    "therapeutic", "direction", "insert", "inserts", "brochures", "brochure",
    "cards", "card", "statement", "setting", "file", "cover", "page", "date",
    "mcg", "summary", "baseline", "endpoint", "table", "rate", "mean", "group",
    "groups", "overall", "analysis", "guidance", "reported", "considered",
    "provided", "issues", "approximately", "doctor", "refer", "section",
    "annex", "annexe", "appendix", "figure", "fig", "presentation", "presentations",
    "vaccine", "vaccin", "biologic", "radiopharmaceutical", "radiopharmaceutiques",
    "drugs", "générale", "validation", "traitements", "types", "requis",
    "responsibility", "therapies", "genetic", "veterinary", "directorate",
    "toute", "ministre", "minister", "dépliant", "échantillons", "intellectuelle",
    "jointe", "emballage", "fiche", "forme", "visés", "paragraphe", "utilisées",
    "déclaration", "utilisée", "modification", "utiliser", "nécessite",
    "changement", "destinée", "abrégée", "abbreviated", "exigences", "drogues",
    "finales", "abrégées", "soumetttre", "obligations", "subsection", "require",
    "specified", "respect", "matters", "change", "products", "supplements",
    "package", "prévu", "commencer", "indiquant", "laquelle", "destinées",
    "office", "connection", "nouvelle",
    "événements", "indésirables", "études", "cliniques", "étant", "vitro",
    "étapes", "virus", "outre", "rapports", "copies", "règlements", "aliments",
    "article", "présente", "soit", "thérapeutiques", "suppléments",
    "drogue", "produit", "produits", "santé", "présentation", "présentations",
    "fabricant", "étiquette", "étiquettes", "compris", "alinéa", "conformément",
    "rappelons", "définitive", "parvenir", "prions", "soumettre", "dossier",
    "médicament", "médicaments", "propriété", "couverture", "biologiques",
    "sommaire", "examen", "médecin",
    "sabourin", "barbara", "supriya", "sharma", "peterson", "robert",
    "sophie", "laurent", "alfred", "celia", "lourenco", "renaud", "louis",
    "omer", "boudreau", "phd", "mph", "frcpc", "bpa", "orl", "tab",
    "sipd", "trd", "crm", "attn", "signé", "signed", "verso",
    "fabriquant", "expliquées", "notification", "formule",
}

ALL_STOPWORDS = stop_en | stop_fr | domain_stopwords

def custom_analyzer(text):
    if not isinstance(text, str):
        return []
    s = text.lower()
    tokens = re.findall(r"[a-zàâäçéèêëîïôöùûüÿñæœ]{3,}", s)
    out = []
    for t in tokens:
        if t in ALL_STOPWORDS:
            continue
        if t.isdigit():
            continue
        out.append(t)
    return out

spark = SparkSession.builder.getOrCreate()

df = (
    spark.table(SOURCE_TABLE)
         .select("path", "final_text", "embedding")
         .filter(F.col("embedding").isNotNull())
)

print(f"Docs loaded (embedding not null): {df.count():,}")

try:
    df_len = df.withColumn("emb_len", F.size("embedding"))
    print("Embedding length distribution (top 20):")
    df_len.groupBy("emb_len").count().orderBy(F.desc("count")).show(20, False)
except Exception as e:
    print("Could not compute embedding lengths in Spark (embedding may not be array). Will compute in pandas.")
    df_len = None

pdf = df.toPandas()

pdf["emb_len"] = pdf["embedding"].apply(lambda x: len(x) if x is not None else -1)

len_counts = pdf["emb_len"].value_counts()
print("Embedding length distribution in pandas (top 10):")
print(len_counts.head(10))

if EMBED_LEN_FORCE is not None:
    target_len = int(EMBED_LEN_FORCE)
else:
    target_len = int(len_counts.index[0])

print(f"Keeping embeddings of length = {target_len}")

excluded = pdf[pdf["emb_len"] != target_len][["path", "emb_len"]].copy()
print(f"Excluded due to wrong length: {len(excluded):,}")

pdf = pdf[pdf["emb_len"] == target_len].reset_index(drop=True)
print(f"Docs after length filter: {len(pdf):,}")

if len(pdf) == 0:
    raise ValueError("No documents left after filtering by embedding length.")

X = np.vstack(pdf["embedding"].values).astype(np.float32)
print(f"Embedding matrix: {X.shape}")

zero_mask = (X == 0).all(axis=1)
print(f"Zero embeddings removed: {int(zero_mask.sum())}")
pdf = pdf[~zero_mask].reset_index(drop=True)
X = X[~zero_mask]

if X.shape[0] == 0:
    raise ValueError("All embeddings were zero vectors after filtering. Nothing to cluster.")

X = normalize(X)

print(f"\nRunning PCA ({PCA_DIMS} dims)...")
pca = PCA(n_components=PCA_DIMS, random_state=42)
X_pca = pca.fit_transform(X)
print(f"PCA shape: {X_pca.shape}")

print(f"Running UMAP ({UMAP_DIMS} dims for clustering)...")
reducer = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    n_components=UMAP_DIMS,
    min_dist=UMAP_MIN_DIST,
    metric="cosine",
    random_state=42,
    low_memory=True
)
X_umap = reducer.fit_transform(X_pca)
print(f"UMAP shape: {X_umap.shape}")

print("\nRunning HDBSCAN...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_method="eom",
    prediction_data=True
)
labels = clusterer.fit_predict(X_umap)
pdf["cluster"] = labels
pdf["was_noise"] = (labels == -1)

n_noise    = int((labels == -1).sum())
n_total    = int(len(labels))
n_clusters = int(len(set(labels)) - (1 if -1 in labels else 0))
print(f"Clusters found: {n_clusters}")
print(f"Noise docs:     {n_noise} ({(n_noise/n_total):.1%})")

if REASSIGN_NOISE and n_noise > 0 and n_clusters > 0:
    print("\nReassigning noise docs to nearest cluster centroid...")

    labels_arr = pdf["cluster"].values

    centroids = {}
    for cid in sorted(set(labels_arr)):
        if cid == -1:
            continue
        idx = np.where(labels_arr == cid)[0]
        cvec = X[idx].mean(axis=0)
        cvec = cvec / (np.linalg.norm(cvec) + 1e-12)
        centroids[int(cid)] = cvec

    centroid_ids = np.array(list(centroids.keys()), dtype=np.int32)
    centroid_mat = np.vstack([centroids[c] for c in centroid_ids]).astype(np.float32)

    noise_idx = np.where(labels_arr == -1)[0]
    sims = X[noise_idx] @ centroid_mat.T
    best = np.argmax(sims, axis=1)
    pdf.loc[noise_idx, "cluster"] = centroid_ids[best]

    print(f"Noise after reassignment: {int((pdf['cluster'] == -1).sum())}")

print("\nBuilding TF-IDF topic labels...")

vectorizer = TfidfVectorizer(
    analyzer=custom_analyzer,
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE
)
tfidf = vectorizer.fit_transform(pdf["final_text"].astype(str).values)
terms = np.array(vectorizer.get_feature_names_out())

cluster_label_map = {}
for cid in sorted(set(pdf["cluster"].values)):
    if cid == -1:
        continue
    idx = np.where(pdf["cluster"].values == cid)[0]
    mean_tfidf = np.asarray(tfidf[idx].mean(axis=0)).ravel()
    top_idx = np.argsort(mean_tfidf)[-TOP_WORDS:][::-1]
    cluster_label_map[int(cid)] = ", ".join(terms[top_idx])

pdf["cluster_keywords"] = pdf["cluster"].map(lambda c: cluster_label_map.get(int(c), "noise"))

print("\nTop keywords per cluster:")
for cid, kw in sorted(cluster_label_map.items()):
    count = int((pdf["cluster"] == cid).sum())
    print(f"  Cluster {cid:>3} ({count:>6} docs): {kw}")

print(f"\nRunning UMAP 3D for visualization...")
reducer_3d = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    n_components=UMAP_VIZ_DIMS,
    min_dist=UMAP_VIZ_MIN_DIST,
    metric="cosine",
    random_state=42,
    low_memory=True
)
X_3d = reducer_3d.fit_transform(X_pca)

pdf["x"] = X_3d[:, 0]
pdf["y"] = X_3d[:, 1]
pdf["z"] = X_3d[:, 2]

def sample_group(g):
    n = max(1, int(np.ceil(len(g) * VIZ_SAMPLE_FRAC)))
    return g.sample(n=min(n, len(g)), random_state=42)

plot_df = (
    pdf.groupby("cluster", group_keys=False)
       .apply(sample_group)   
       .reset_index(drop=True)
)

plot_df["preview"] = plot_df["final_text"].astype(str).str.replace("\n", " ").str[:100]
plot_df["cluster_str"] = "Cluster " + plot_df["cluster"].astype(str)
plot_df["rescued"] = plot_df["was_noise"].map({True: "Yes (rescued noise)", False: "No"})

print(f"Sampled {len(plot_df):,} docs for visualization")

fig = px.scatter_3d(
    plot_df,
    x="x", y="y", z="z",
    color="cluster_str",
    hover_data={
        "cluster_str": True,
        "cluster_keywords": True,
        "rescued": True,
        "preview": True,
        "x": False, "y": False, "z": False,
    },
    title=f"NOC Document Clusters — {n_clusters} clusters · {len(pdf):,} docs · emb_len={target_len}",
    labels={"cluster_str": "Cluster"},
    opacity=0.65,
    height=850,
)
fig.update_traces(marker=dict(size=2.5))
fig.update_layout(
    scene=dict(
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        zaxis_title="UMAP-3",
        bgcolor="rgb(15,17,23)",
    ),
    paper_bgcolor="rgb(15,17,23)",
    font_color="white",
    legend_title_text="Cluster",
)

try:
    os.makedirs("/dbfs/tmp", exist_ok=True)
    html_path = "/dbfs/tmp/cluster_viz_3d.html"
    fig.write_html(html_path)
    print(f"3D viz saved: {html_path}")
except Exception:
    html_path = "/tmp/cluster_viz_3d.html"
    fig.write_html(html_path)
    print(f"3D viz saved (local): {html_path}")
    print("Copy to DBFS: dbutils.fs.cp('file:/tmp/cluster_viz_3d.html', 'dbfs:/tmp/cluster_viz_3d.html')")

displayHTML(fig.to_html(full_html=False, include_plotlyjs="cdn"))

out_pdf = pdf[["path", "cluster", "cluster_keywords", "was_noise"]].copy()

if len(out_pdf) == 0:
    raise ValueError("No rows to save (out_pdf is empty).")

out_sdf = spark.createDataFrame(out_pdf)
out_sdf.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(OUT_TABLE)
print(f"\nResults saved → {OUT_TABLE}")

spark.table(OUT_TABLE) \
     .groupBy("cluster", "cluster_keywords") \
     .agg(F.count("*").alias("doc_count")) \
     .orderBy(F.desc("doc_count")) \
     .show(200, truncate=False)

In [0]:
%pip install -q umap-learn hdbscan scikit-learn plotly

import re
import os
import numpy as np
import pandas as pd
import plotly.express as px

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.feature_extraction import text as sk_text
from sklearn.feature_extraction.text import TfidfVectorizer

import umap
import hdbscan


SOURCE_TABLE  = "noc_documents_multilingual"
OUT_TABLE     = "noc_documents_clustering5"

PCA_DIMS          = 50
UMAP_DIMS         = 15
UMAP_NEIGHBORS    = 30
UMAP_MIN_DIST     = 0.0

UMAP_VIZ_DIMS     = 3
UMAP_VIZ_MIN_DIST = 0.1

MIN_CLUSTER_SIZE  = 50
MIN_SAMPLES       = 5

TOP_WORDS         = 12
MAX_FEATURES      = 5000
MIN_DF            = 10
MAX_DF            = 0.60      
SUBLINEAR_TF      = True      

REASSIGN_NOISE    = False
VIZ_SAMPLE_FRAC   = 0.5

RANDOM_STATE      = 42


stop_en = set(sk_text.ENGLISH_STOP_WORDS)

stop_fr = {
    "alors","au","aucuns","aussi","autre","avant","avec","avoir","bon","car",
    "ce","cela","ces","ceux","chaque","ci","comme","comment","dans","des","du",
    "dedans","dehors","depuis","devrait","doit","donc","dos","début","elle",
    "elles","en","encore","essai","est","et","eu","fait","faites","fois","font",
    "force","haut","hors","ici","il","ils","je","juste","la","le","les","leur",
    "là","ma","maintenant","mais","mes","mine","moins","mon","mot","même","ni",
    "nommés","notre","nous","nouveaux","ou","où","par","parce","parole","pas",
    "personnes","peut","peu","pièce","plupart","pour","pourquoi","quand","que",
    "quel","quelle","quelles","quels","qui","sa","sans","ses","seulement","si",
    "sien","son","sont","sous","soyez","sujet","sur","ta","tandis","tellement",
    "tels","tes","ton","tous","tout","trop","très","tu","valeur","voie","voient",
    "vont","votre","vous","vu","ça","étaient","état","étions","été","être",
    "un","une","aux","toute","toutes","tous","tout","autre","autres","même",
    "aussi","alors","après","avant","comme","entre","sans","sous","lors","selon",
    "faire","dit","sont","ont","était","ainsi","cas","doit","peut","étaient",
    "également","émis","éléments","jusqu","cours","fois","trois","deux","quatre",
    "décrites","portant","conséquent","jours","veuillez","effets","motifs",
    "aujourd","hui",
}

domain_stopwords = {
    "drug", "canada", "health", "product", "notice", "submission", "submissions",
    "authorization", "manufacturer", "minister", "office", "bureau", "label",
    "labelling", "version", "final", "new", "used", "use", "including", "included",
    "proposed", "requested", "request", "direct", "intended", "accordance",
    "paragraph", "reminded", "connection", "basis", "document", "documents",
    "specimens", "specimen", "submit", "labels", "intellectual", "property",
    "therapeutic", "direction", "insert", "inserts", "brochures", "brochure",
    "cards", "card", "statement", "setting", "file", "cover", "page", "date",
    "mcg", "summary", "baseline", "endpoint", "table", "rate", "mean", "group",
    "groups", "overall", "analysis", "guidance", "reported", "considered",
    "provided", "issues", "approximately", "doctor", "refer", "section",
    "annex", "annexe", "appendix", "figure", "fig", "presentation", "presentations",
    "vaccine", "vaccin", "biologic", "radiopharmaceutical", "radiopharmaceutiques",
    "drugs", "générale", "validation", "traitements", "types", "requis",
    "responsibility", "therapies", "genetic", "veterinary", "directorate",
    "toute", "ministre", "dépliant", "échantillons", "intellectuelle",
    "jointe", "emballage", "fiche", "forme", "visés", "paragraphe", "utilisées",
    "déclaration", "utilisée", "modification", "utiliser", "nécessite",
    "changement", "destinée", "abrégée", "abbreviated", "exigences", "drogues",
    "finales", "abrégées", "soumetttre", "obligations", "subsection", "require",
    "specified", "respect", "matters", "change", "products", "supplements",
    "package", "prévu", "commencer", "indiquant", "laquelle", "destinées",
    "office", "connection", "nouvelle",
    "événements", "indésirables", "études", "cliniques", "étant", "vitro",
    "étapes", "virus", "outre", "rapports", "copies", "règlements", "aliments",
    "article", "présente", "soit", "thérapeutiques", "suppléments",
    "drogue", "produit", "produits", "santé", "présentation", "présentations",
    "fabricant", "étiquette", "étiquettes", "compris", "alinéa", "conformément",
    "rappelons", "définitive", "parvenir", "prions", "soumettre", "dossier",
    "médicament", "médicaments", "propriété", "couverture", "biologiques",
    "sommaire", "examen", "médecin",
    "sabourin", "barbara", "supriya", "sharma", "peterson", "robert",
    "sophie", "laurent", "alfred", "celia", "lourenco", "renaud", "louis",
    "omer", "boudreau", "phd", "mph", "frcpc", "bpa", "orl", "tab",
    "sipd", "trd", "crm", "attn", "signé", "signed", "verso",
    "fabriquant", "expliquées", "notification", "formule",
}

ALL_STOPWORDS = stop_en | stop_fr | domain_stopwords

TOKEN_RE = re.compile(r"[a-zàâäçéèêëîïôöùûüÿñæœ]{3,}")

def custom_analyzer_1_2grams(text):
    if not isinstance(text, str):
        return []
    s = text.lower()
    toks = TOKEN_RE.findall(s)

    clean = []
    for t in toks:
        if t in ALL_STOPWORDS:
            continue
        if t.isdigit():
            continue
        clean.append(t)

    # unigrams + bigrams
    grams = list(clean)
    for i in range(len(clean) - 1):
        grams.append(clean[i] + " " + clean[i+1])

    return grams


spark = SparkSession.builder.getOrCreate()

df = (
    spark.table(SOURCE_TABLE)
         .select("path", "final_text", "embedding")
         .filter(F.col("embedding").isNotNull())
)

print(f"Docs loaded: {df.count():,}")

pdf = df.toPandas()
X = np.vstack(pdf["embedding"].values).astype(np.float32)
print(f"Embedding matrix: {X.shape}")

zero_mask = (X == 0).all(axis=1)
print(f"Zero embeddings removed: {int(zero_mask.sum())}")

if zero_mask.any():
    pdf = pdf[~zero_mask].reset_index(drop=True)
    X = X[~zero_mask]

X = normalize(X)

print(f"\nRunning PCA ({PCA_DIMS} dims)...")
pca = PCA(n_components=PCA_DIMS, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
print(f"PCA shape: {X_pca.shape}")

print(f"Running UMAP ({UMAP_DIMS} dims for clustering)...")
reducer = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    n_components=UMAP_DIMS,
    min_dist=UMAP_MIN_DIST,
    metric="cosine",
    random_state=RANDOM_STATE,
    low_memory=True
)
X_umap = reducer.fit_transform(X_pca)
print(f"UMAP shape: {X_umap.shape}")

print("\nRunning HDBSCAN...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_method="eom",
    prediction_data=True
)
labels = clusterer.fit_predict(X_umap)

pdf["cluster"] = labels.astype(int)
pdf["was_noise"] = (labels == -1)

n_noise    = int((labels == -1).sum())
n_total    = int(len(labels))
n_clusters = int(len(set(labels)) - (1 if -1 in labels else 0))

print(f"Clusters found: {n_clusters}")
print(f"Noise docs:     {n_noise} ({(n_noise/n_total):.1%})")

if REASSIGN_NOISE and n_noise > 0 and n_clusters > 0:
    print("\nReassigning noise docs to nearest cluster centroid...")

    labels_arr = pdf["cluster"].values

    centroids = {}
    for cid in sorted(set(labels_arr)):
        if cid == -1:
            continue
        idx = np.where(labels_arr == cid)[0]
        cvec = X[idx].mean(axis=0)
        cvec /= (np.linalg.norm(cvec) + 1e-12)
        centroids[int(cid)] = cvec

    centroid_ids = np.array(list(centroids.keys()), dtype=np.int32)
    centroid_mat = np.vstack([centroids[c] for c in centroid_ids]).astype(np.float32)

    noise_idx = np.where(labels_arr == -1)[0]
    sims = X[noise_idx] @ centroid_mat.T
    best = np.argmax(sims, axis=1)
    pdf.loc[noise_idx, "cluster"] = centroid_ids[best]

    print(f"Noise after reassignment: {int((pdf['cluster'].values == -1).sum())}")

print("\nBuilding TF-IDF topic labels...")

vectorizer = TfidfVectorizer(
    analyzer=custom_analyzer_1_2grams,  
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    max_df=MAX_DF,                     
    sublinear_tf=SUBLINEAR_TF            
)

tfidf = vectorizer.fit_transform(pdf["final_text"].astype(str).values)
terms = np.array(vectorizer.get_feature_names_out())

cluster_label_map = {}
for cid in sorted(set(pdf["cluster"].values)):
    if cid == -1:
        continue
    idx = np.where(pdf["cluster"].values == cid)[0]
    mean_tfidf = np.asarray(tfidf[idx].mean(axis=0)).ravel()
    top_idx = np.argsort(mean_tfidf)[-TOP_WORDS:][::-1]
    cluster_label_map[int(cid)] = ", ".join(terms[top_idx])

pdf["cluster_keywords"] = pdf["cluster"].map(lambda c: cluster_label_map.get(int(c), "noise"))

print("\nTop keywords per cluster:")
for cid, kw in sorted(cluster_label_map.items()):
    count = int((pdf["cluster"].values == cid).sum())
    print(f"  Cluster {cid:>3} ({count:>5} docs): {kw}")

print(f"\nRunning UMAP {UMAP_VIZ_DIMS}D for visualization...")
reducer_3d = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    n_components=UMAP_VIZ_DIMS,
    min_dist=UMAP_VIZ_MIN_DIST,
    metric="cosine",
    random_state=RANDOM_STATE,
    low_memory=True
)
X_3d = reducer_3d.fit_transform(X_pca)

pdf["x"] = X_3d[:, 0]
pdf["y"] = X_3d[:, 1]
pdf["z"] = X_3d[:, 2]

sample_idx = (
    pdf.groupby("cluster", group_keys=False)
       .apply(lambda g: g.sample(frac=VIZ_SAMPLE_FRAC, random_state=RANDOM_STATE))
       .index
)
plot_df = pdf.loc[sample_idx].copy().reset_index(drop=True)

plot_df["preview"] = plot_df["final_text"].astype(str).str[:120].str.replace("\n", " ", regex=False)
plot_df["cluster_str"] = "Cluster " + plot_df["cluster"].astype(str)
plot_df["rescued"] = plot_df["was_noise"].map({True: "Yes (noise)", False: "No"})

print(f"Sampled {len(plot_df):,} docs for visualization")

fig = px.scatter_3d(
    plot_df,
    x="x", y="y", z="z",
    color="cluster_str",
    hover_data={
        "cluster_str": True,
        "cluster_keywords": True,
        "rescued": True,
        "preview": True,
        "x": False, "y": False, "z": False,
    },
    title=f"NOC Document Clusters — {n_clusters} clusters · {len(pdf):,} docs",
    labels={"cluster_str": "Cluster"},
    opacity=0.65,
    height=850,
)
fig.update_traces(marker=dict(size=2.5))
fig.update_layout(
    scene=dict(
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        zaxis_title="UMAP-3",
        bgcolor="rgb(15,17,23)",
    ),
    paper_bgcolor="rgb(15,17,23)",
    font_color="white",
    legend_title_text="Cluster",
)

try:
    os.makedirs("/dbfs/tmp", exist_ok=True)
    html_path = "/dbfs/tmp/cluster_viz_3d.html"
    fig.write_html(html_path)
    print(f"3D viz saved: {html_path}")
except Exception:
    html_path = "/tmp/cluster_viz_3d.html"
    fig.write_html(html_path)
    print(f"3D viz saved (local): {html_path}")
    print("Copy to DBFS: dbutils.fs.cp('file:/tmp/cluster_viz_3d.html', 'dbfs:/tmp/cluster_viz_3d.html')")

displayHTML(fig.to_html(full_html=False, include_plotlyjs="cdn"))

out_pdf = pdf[["path", "cluster", "cluster_keywords", "was_noise"]].copy()
out_sdf = spark.createDataFrame(out_pdf)

out_sdf.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(OUT_TABLE)
print(f"\nResults saved → {OUT_TABLE}")

spark.table(OUT_TABLE) \
     .groupBy("cluster", "cluster_keywords") \
     .agg(F.count("*").alias("doc_count")) \
     .orderBy(F.desc("doc_count")) \
     .show(200, truncate=False)

In [0]:
from pyspark.sql import functions as F

filtered_df = (
    spark.table("noc_documents_multilingual")
        .filter(F.lower(F.col("final_text")).contains("estradiol "))
        .select("path", "final_text")
)

filtered_df.display()

row_count = filtered_df.count()
print("Total documents containing 'ketorolac':", row_count)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df = spark.table("noc_documents_clustering")

cluster_df = (
    df.select("cluster", "cluster_keywords")
    .dropDuplicates(["cluster"])
    .orderBy("cluster")
    .toPandas()
)

print(f"Total clusters to name: {len(cluster_df)}")

KEYWORD_RULES = [
    # ── Vaccines & Immunologicals ──
    (["influenza", "haemagglutinin", "hemagglutinine", "strain", "souche", "yamagata", "victoria"], "Influenza Vaccine"),
    (["nimenrix", "bexsero", "meningococcal", "meningitidis", "polysaccharide", "meningocoque"], "Meningococcal Vaccine"),
    (["gardasil", "cervarix", "papillomavirus", "hpv", "vph"], "HPV Vaccine"),
    (["rotavirus", "vaccination", "nimenrix", "bexsero"], "Pediatric Vaccines"),
    (["pneumococcal", "serotype", "pneumovax"], "Pneumococcal Vaccine"),
    (["toxoid", "tetanus", "pertussis", "diphtheria", "poliovirus", "haemophilus", "tétanique"], "Combination Vaccine (DTaP/IPV)"),
    (["hepatitis", "inactivated", "antigen", "immunizing", "active"], "Hepatitis Vaccine"),
    (["poliovirus", "anatoxine", "tétanique", "pertactin", "filamentous"], "Polio/DTP Vaccine"),

    # ── Biologics & Biosimilars ──
    (["trastuzumab", "herceptin", "pertuzumab", "perjeta"], "Trastuzumab/Herceptin (Breast Cancer)"),
    (["bevacizumab", "avastin"], "Bevacizumab/Avastin (Cancer)"),
    (["nivolumab", "opdivo", "pembrolizumab", "keytruda"], "Checkpoint Inhibitors (Nivolumab/Pembrolizumab)"),
    (["aflibercept", "eylea", "ranibizumab", "lucentis"], "Anti-VEGF Ophthalmic Biologics"),
    (["adalimumab", "infliximab", "inflectra", "remicade", "cimzia"], "Anti-TNF Biologics"),
    (["cosentyx", "secukinumab"], "Secukinumab/Cosentyx"),
    (["tocilizumab", "actemra"], "Tocilizumab/Actemra"),
    (["obinutuzumab", "gazyva"], "Obinutuzumab/Gazyva"),
    (["liraglutide", "victoza"], "Liraglutide/Victoza"),
    (["interferon", "beta", "peginterferon", "alfa"], "Interferon (Beta/Alpha)"),
    (["immune", "globulin", "immunoglobulin", "immunoglobuline"], "Immune Globulin"),
    (["factor", "coagulation", "recombinant", "facteur", "pwso"], "Coagulation Factors (Biologics)"),
    (["insulin", "insuline", "nordisk", "novo", "glargine", "lispro"], "Insulin Products"),
    (["enoxaparin", "anticoagulant", "sodique", "sodium"], "Enoxaparin/Anticoagulant Injectables"),

    # ── Oncology ──
    (["imatinib", "gleevec", "mesylate"], "Imatinib/Gleevec (CML)"),
    (["bortezomib", "velcade", "mannitol"], "Bortezomib/Velcade (Myeloma)"),
    (["lenalidomide", "pomalidomide", "celgene", "revlimid"], "Lenalidomide/Pomalidomide (Myeloma)"),
    (["docetaxel", "doxorubicin", "paclitaxel", "cabazitaxel", "jevtana"], "Taxane/Anthracycline Chemotherapy"),
    (["irinotecan", "cisplatin"], "Irinotecan/Cisplatin Chemotherapy"),
    (["abiraterone", "zytiga", "acetate"], "Abiraterone/Zytiga (Prostate)"),
    (["palbociclib", "ibrance"], "Palbociclib/Ibrance (Breast Cancer)"),
    (["letrozole", "femara", "anastrozole", "arimidex", "aromatase"], "Aromatase Inhibitors (Breast Cancer)"),
    (["dabrafenib", "trametinib", "mekinist", "tafinlar"], "BRAF/MEK Inhibitors"),
    (["antineoplastic", "intravenous", "injection"], "Antineoplastic Injectables"),
    (["gleevec", "sutent", "tasigna", "velcade", "avastin", "lmc"], "NOC-C Oncology Conditions"),
    (["methotrexate", "méthotrexate"], "Methotrexate"),
    (["deferasirox", "exjade"], "Deferasirox/Exjade (Iron Chelation)"),

    # ── Cardiovascular ──
    (["atorvastatin", "lipitor", "atorvastatine"], "Atorvastatin/Lipitor (Statin)"),
    (["rosuvastatin", "crestor", "rosuvastatine"], "Rosuvastatin/Crestor (Statin)"),
    (["simvastatin"], "Simvastatin (Statin)"),
    (["pravastatin"], "Pravastatin (Statin)"),
    (["lipid", "regulator", "statin", "metabolism", "rosuvastatin", "atorvastatin", "simvastatin"], "Lipid Regulators / Statins"),
    (["ezetimibe"], "Ezetimibe (Cholesterol)"),
    (["rivaroxaban", "xarelto"], "Rivaroxaban/Xarelto (Anticoagulant)"),
    (["apixaban", "eliquis", "dabigatran"], "Apixaban/Eliquis (Anticoagulant)"),
    (["clopidogrel", "ticagrelor", "brilinta", "platelet", "aggregation"], "Antiplatelet Agents"),
    (["amlodipine", "norvasc", "besylate"], "Amlodipine/Norvasc (CCB)"),
    (["atenolol", "metoprolol", "blocking"], "Beta Blockers (Atenolol/Metoprolol)"),
    (["telmisartan", "boehringer"], "Telmisartan (ARB)"),
    (["candesartan", "atacand", "cilexetil"], "Candesartan/Atacand (ARB)"),
    (["irbesartan", "valsartan", "losartan", "olmesartan"], "ARBs (Irbesartan/Valsartan/Losartan)"),
    (["perindopril", "coversyl", "erbumine"], "Perindopril/Coversyl (ACE Inhibitor)"),
    (["lisinopril", "enalapril", "ramipril", "converting", "enzyme", "angiotensin"], "ACE Inhibitors"),
    (["amiodarone"], "Amiodarone (Antiarrhythmic)"),
    (["sildenafil", "citrate"], "Sildenafil (ED/PAH)"),
    (["ambrisentan", "olmesartan"], "Pulmonary Arterial Hypertension"),

    # ── Diabetes ──
    (["metformin", "metformine", "hydrochloride", "aventis", "sanofi"], "Metformin (Antidiabetic)"),
    (["sitagliptin", "januvia", "sitagliptine", "dpp"], "Sitagliptin/Januvia (DPP-4 Inhibitor)"),
    (["dapagliflozin", "forxiga", "dapagliflozine"], "Dapagliflozin/Forxiga (SGLT2 Inhibitor)"),
    (["canagliflozin", "invokana", "jardiance"], "SGLT2 Inhibitors (Canagliflozin/Empagliflozin)"),
    (["pioglitazone", "antihyperglycemic"], "Pioglitazone (TZD Antidiabetic)"),
    (["saxagliptin"], "Saxagliptin (DPP-4 Inhibitor)"),

    # ── CNS / Psychiatry ──
    (["fluoxetine", "antiobsessional", "antidepressant", "eli", "lilly"], "Fluoxetine (SSRI)"),
    (["sertraline", "zoloft", "hydrochloride", "pfizer"], "Sertraline (SSRI)"),
    (["paroxetine", "hemihydrate", "glaxosmithkline"], "Paroxetine (SSRI)"),
    (["escitalopram", "citalopram", "lundbeck", "cipralex"], "Escitalopram/Citalopram (SSRI)"),
    (["venlafaxine", "desvenlafaxine", "pristiq", "wyeth"], "Venlafaxine/Desvenlafaxine (SNRI)"),
    (["duloxetine", "cymbalta", "duloxétine"], "Duloxetine/Cymbalta (SNRI)"),
    (["bupropion", "valeant", "elzear"], "Bupropion (Antidepressant)"),
    (["mirtazapine", "organon"], "Mirtazapine (Antidepressant)"),
    (["quetiapine", "seroquel", "aripiprazole", "abilify"], "Atypical Antipsychotics (Quetiapine/Aripiprazole)"),
    (["olanzapine", "zyprexa", "antipsychotic", "disintegrating"], "Olanzapine/Zyprexa (Antipsychotic)"),
    (["risperidone", "antipsychotic", "janssen", "ortho"], "Risperidone (Antipsychotic)"),
    (["pregabalin", "lyrica", "prégabaline"], "Pregabalin/Lyrica (Neuropathic Pain)"),
    (["gabapentin", "neurontin"], "Gabapentin/Neurontin (Neuropathic Pain)"),
    (["methylphenidate", "lisdexamfetamine", "vyvanse", "amphetamine", "adhd"], "ADHD Medications"),
    (["topiramate", "lamotrigine", "antiepileptic"], "Antiepileptics (Topiramate/Lamotrigine)"),
    (["levetiracetam", "ucb"], "Levetiracetam (Antiepileptic)"),
    (["carbamazepine", "valproic", "acid", "sodium"], "Carbamazepine/Valproate (Antiepileptic)"),
    (["fingolimod", "gilenya", "dimethyl", "fumarate", "teriflunomide", "aubagio"], "MS Disease-Modifying Therapies"),
    (["memantine", "ebixa", "donepezil", "rivastigmine", "alzheimer"], "Alzheimer's / Dementia Drugs"),
    (["pramipexole", "dopamine", "agonist"], "Pramipexole (Parkinson's)"),
    (["carbidopa", "levodopa"], "Levodopa/Carbidopa (Parkinson's)"),
    (["zopiclone", "sedative"], "Zopiclone (Hypnotic)"),
    (["varenicline", "champix"], "Varenicline/Champix (Smoking Cessation)"),

    # ── Pain / Opioids ──
    (["tramadol", "acetaminophen", "tramacet"], "Tramadol/Acetaminophen (Tramacet)"),
    (["oxycodone", "purdue", "prolongée", "extended"], "Oxycodone ER (Opioid)"),
    (["fentanyl", "transdermal", "patch", "opioid"], "Fentanyl Patch (Opioid)"),
    (["hydromorphone", "naloxone", "opioid", "intravenous"], "Hydromorphone/Naloxone (Opioid)"),
    (["buprenorphine", "naloxone", "sublingual"], "Buprenorphine/Naloxone (Opioid Dependence)"),
    (["ketorolac", "tromethamine", "intramuscular"], "Ketorolac (NSAID Injectable)"),
    (["ibuprofen", "advil", "analgesic", "antipyretic"], "Ibuprofen/Advil (OTC NSAID)"),
    (["naproxen", "sodium"], "Naproxen (NSAID)"),
    (["diclofenac", "voltaren", "emulgel", "nsaid"], "Diclofenac/Voltaren (NSAID)"),
    (["celecoxib", "celebrex", "nsaid"], "Celecoxib/Celebrex (COX-2 Inhibitor)"),
    (["acetylsalicylic", "asa", "aspirin"], "Acetylsalicylic Acid / ASA"),
    (["sumatriptan", "rizatriptan", "zolmitriptan", "migraine", "agonist"], "Triptans (Migraine)"),

    # ── Respiratory ──
    (["salbutamol", "inhalation", "metered", "bronchodilator", "fluticasone"], "Salbutamol/Fluticasone Inhalers"),
    (["fluticasone", "vilanterol", "breo", "ellipta", "copd", "mpoc"], "Fluticasone/Vilanterol (COPD)"),
    (["montelukast", "chewable", "sodium", "antagonist"], "Montelukast (Asthma/Allergic Rhinitis)"),
    (["nasal", "spray", "fluticasone", "metered", "furoate"], "Nasal Sprays (Fluticasone)"),

    # ── GI / GU ──
    (["pantoprazole", "pantoloc", "sesquihydrate", "nycomed", "proton"], "Pantoprazole/Pantoloc (PPI)"),
    (["omeprazole", "esomeprazole", "magnesium", "astrazeneca"], "Omeprazole/Esomeprazole (PPI)"),
    (["lansoprazole", "takeda", "delayed"], "Lansoprazole (PPI)"),
    (["domperidone", "prucalopride", "maleate"], "Domperidone/Prucalopride (Motility)"),
    (["ranitidine", "histamine", "antagonist", "receptor"], "Ranitidine (H2 Blocker)"),
    (["famotidine", "histamine", "antagonist"], "Famotidine (H2 Blocker)"),
    (["ondansetron", "zofran", "antagonist"], "Ondansetron/Zofran (Antiemetic)"),
    (["solifenacin", "vesicare", "astellas"], "Solifenacin/Vesicare (Overactive Bladder)"),
    (["mycophenolate", "roche", "hoffmann"], "Mycophenolate (Immunosuppressant)"),
    (["tacrolimus", "cyclosporine", "astellas"], "Tacrolimus/Cyclosporine (Transplant)"),

    # ── Ophthalmology ──
    (["latanoprost", "timolol", "ophthalmic", "dorzolamide", "ophtalmique"], "Ophthalmic Solutions (Glaucoma)"),

    # ── Endocrine / Bone ──
    (["alendronate", "risedronate", "bone", "regulator", "disodium"], "Bisphosphonates (Alendronate/Risedronate)"),
    (["finasteride", "alpha", "inhibitor"], "Finasteride (BPH/Hair Loss)"),
    (["estradiol", "ethinyl", "contraceptive", "levonorgestrel"], "Oral Contraceptives / Estrogen"),
    (["progesterone"], "Progesterone"),
    (["calcitriol", "amgen"], "Calcitriol (Vitamin D Analog)"),

    # ── Anti-infectives ──
    (["valacyclovir", "antiviral", "glaxosmithkline"], "Valacyclovir (Antiviral)"),
    (["fluconazole", "antifungal", "pfizer"], "Fluconazole (Antifungal)"),
    (["voriconazole", "antifungal", "scott"], "Voriconazole (Antifungal)"),
    (["azithromycin", "dihydrate", "pfizer", "antibiotic"], "Azithromycin (Antibiotic)"),
    (["ciprofloxacin", "cipro", "bayer", "antibacterial"], "Ciprofloxacin/Cipro (Fluoroquinolone)"),
    (["levofloxacin", "hemihydrate", "ortho"], "Levofloxacin (Fluoroquinolone)"),
    (["moxifloxacin", "bayer", "hydrochloride"], "Moxifloxacin (Fluoroquinolone)"),
    (["amoxicillin", "clavulanate", "trihydrate"], "Amoxicillin/Clavulanate (Antibiotic)"),
    (["clindamycin", "doxycycline", "clarithromycin"], "Macrolide/Clindamycin Antibiotics"),
    (["vancomycin", "hydrochloride", "powder", "vial"], "Vancomycin (Antibiotic Injectable)"),
    (["cefazolin", "powder", "intravenous", "intramuscular"], "Cefazolin (Cephalosporin)"),
    (["tazobactam", "piperacillin", "tigecycline"], "Piperacillin/Tazobactam (Antibiotic)"),
    (["tenofovir", "emtricitabine", "disoproxil", "gilead", "entecavir"], "Tenofovir/Emtricitabine (HIV/HBV)"),
    (["abacavir", "lamivudine", "viiv"], "Abacavir/Lamivudine (HIV)"),
    (["dolutegravir", "viiv", "rilpivirine"], "Dolutegravir/Rilpivirine (HIV)"),
    (["darunavir", "atazanavir", "ritonavir", "antiretroviral"], "HIV Protease Inhibitors"),
    (["rbv", "galexos", "svr", "hcv", "vhc", "genotype"], "Hepatitis C Treatment (DAA)"),

    # ── Document / Admin Types ──
    (["vétérinaires", "espèce", "species", "cattle", "records", "registres"], "Veterinary Drug Submissions"),
    (["effectiveness", "efficacité", "extraordinary", "exceptionnel", "innocuité", "safety", "usage", "plan"], "NOC with Conditions (Effectiveness)"),
    (["disinfectant", "buccal", "acid"], "Disinfectants / Oral Care"),
    (["patients", "clinical", "studies", "decision", "efficacy", "safety", "sbd"], "Clinical Decision Letters (EN)"),
    (["patients", "chez", "traitement", "décision", "innocuité", "efficacité", "smd"], "Clinical Decision Letters (FR)"),
    (["effects", "pharmacist", "blood", "taking", "medication", "vigilance"], "Consumer Medication Information"),
    (["gleevec", "sutent", "noc", "policy", "conditions", "cml"], "NOC-C Policy Documents"),
    (["intravascular", "solution", "intravenous", "imaging"], "Intravascular/Imaging Agents"),
    (["dexmedetomidine", "precedex", "hospira"], "Dexmedetomidine/Precedex (Sedative)"),
]

def assign_name(keywords_str):
    if not isinstance(keywords_str, str):
        return "Unknown"

    kw = keywords_str.lower()

    for keyword_list, name in KEYWORD_RULES:
        if any(k in kw for k in keyword_list):
            return name

    first_kw = kw.split(",")[0].strip()
    return first_kw.title() if first_kw else "Unlabeled"


cluster_df["cluster_name"] = cluster_df["cluster_keywords"].apply(assign_name)

print("\nCluster Name Mapping:")
print("-" * 80)
for _, row in cluster_df.iterrows():
    print(f"  Cluster {int(row['cluster']):>3}: {row['cluster_name']}")
    print(f"             Keywords: {row['cluster_keywords'][:80]}")
    print()


unlabeled = cluster_df[cluster_df["cluster_name"].isin(["Unlabeled", "Unknown"])]
if len(unlabeled) > 0:
    print(f"\n  {len(unlabeled)} clusters need manual naming:")
    for _, row in unlabeled.iterrows():
        print(f"  Cluster {int(row['cluster'])}: {row['cluster_keywords']}")
else:
    print(" All clusters named successfully!")


name_lookup = spark.createDataFrame(
    cluster_df[["cluster", "cluster_name"]]
)

df_with_names = (
    spark.table("noc_documents_clustering")
    .join(name_lookup, on="cluster", how="left")
)

df_with_names.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("noc_documents_clustering")

print("\n cluster_name written to noc_documents_clustering")


spark.table("noc_documents_clustering") \
     .groupBy("cluster", "cluster_name") \
     .agg(F.count("*").alias("doc_count")) \
     .orderBy(F.desc("doc_count")) \
     .show(250, truncate=False)

In [0]:
spark.table("noc_documents_clustering") \
     .join(
         spark.table("noc_documents_azure").select("path", "final_text"),
         on="path",
         how="left"
     ) \
     .filter(F.col("cluster") == 34) \
     .select("path", "cluster_keywords", "was_noise", "final_text") \
     .display()